In [1]:
!pip install -q \
langchain \
langchain-community \
langchain-huggingface \
sentence-transformers \
faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 99.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
from google.colab import drive
import os
drive.mount('/content/drive')

os.chdir('/content/drive/MyDrive/ufc-rag-proj')

Mounted at /content/drive


In [3]:
import duckdb
from pathlib import Path
import pandas as pd
from tqdm import tqdm

PROJECT_ROOT = Path("/content/drive/MyDrive/ufc-rag-proj")
DB_PATH = PROJECT_ROOT/ 'data/ufc.duckdb'
DOCS_DIR = PROJECT_ROOT / "documents"

(DOCS_DIR / "fights").mkdir(parents=True, exist_ok=True)
(DOCS_DIR / "fighters").mkdir(parents=True, exist_ok=True)
(DOCS_DIR / "events").mkdir(parents=True, exist_ok=True)

conn = duckdb.connect(str(DB_PATH))
print(conn.execute('SHOW TABLES').fetchdf()['name'].to_list())





['events', 'fight_performances', 'fighters', 'fights']


In [32]:
fights_df = conn.execute(
"""
SELECT
    fi.fight_id,
    e.name as event_name,
    e.date as event_date,
    e.location as event_location,
    weight_class,
    method,
    round,
    time,
    bonus,
    fa.name as winner_name,
    fb.name as loser_name,
    fp_a.sig_strikes_landed as winner_sig_strikes,
    fp_b.sig_strikes_landed as loser_sig_strikes,
    fp_a.knockdowns as winner_knockdowns,
    fp_b.knockdowns as loser_knockdowns,
    fp_a.takedowns_landed as winner_takedowns,
    fp_b.takedowns_landed as loser_takedowns
FROM fights fi
JOIN events e on fi.event_id = e.event_id
JOIN fight_performances fp_a ON fp_a.fight_id = fi.fight_id AND fp_a.is_winner = TRUE
JOIN fighters fa ON fa.fighter_id = fp_a.fighter_id
JOIN fight_performances fp_b ON fp_b.fight_id = fi.fight_id AND fp_b.is_winner = FALSE
JOIN fighters fb ON fb.fighter_id = fp_b.fighter_id


""").fetchdf()

print(len(fights_df))

fights_df.tail(3)

8436


,fight_id,event_name,event_date,event_location,weight_class,method,round,time,bonus,winner_name,loser_name,winner_sig_strikes,loser_sig_strikes,winner_knockdowns,loser_knockdowns,winner_takedowns,loser_takedowns
8433,244285b7fd0ddece,UFC Fight Night: Emmett vs. Vallejos,2026-03-14,"Las Vegas, Nevada, USA",Women's Strawweight,Decision - Unanimous,3.0,5:00,-,PIERA RODRIGUEZ,SAM HUGHES,107,77,0,0,1,0
8434,1ed550f094ca7745,UFC 277: Pena vs. Nunes 2,2022-07-30,"Dallas, Texas, USA",Light Heavyweight,KO/TKO,2.0,3:33,-,NICOLAE NEGUMEREANU,IHOR POTIERIA,51,38,0,0,2,0
8435,eb94412f15c62650,UFC 310: Pantoja vs. Asakura,2024-12-07,"Las Vegas, Nevada, USA",Heavyweight,KO/TKO,1.0,4:51,perf,KENNEDY NZECHUKWU,LUKASZ BRZESKI,29,32,1,0,0,0


In [33]:
def create_fight_document(row):
  doc = f"""
    Event: {row['event_name']}
    Date: {row['event_date'].strftime('%B %d, %Y') if pd.notna(row['event_date']) else "Unknown"}
    Location: {row['event_location']}
    Weight Class: {row['weight_class']}


    Result: {row['winner_name']} defeated {row['loser_name']} via {row['method']}
  """

  if pd.notna(row['round']) and pd.notna(row['time']):
    doc += f"at {row['time']} of round {int(row['round'])}"
  else:
    doc += "."

  doc += f"""
  Key stats:
  - {row['winner_name']} landed {int(row['winner_sig_strikes']) if pd.notna(row['winner_sig_strikes']) else 0} significant strikes
  - {row['loser_name']} landed {int(row['loser_sig_strikes']) if pd.notna(row['loser_sig_strikes']) else 0} significant strikes
  - Knockdowns: {row['winner_name']} {int(row['winner_knockdowns']) if pd.notna(row['winner_knockdowns']) else 0},  {row['loser_name']} {int(row['loser_knockdowns']) if pd.notna(row['loser_knockdowns']) else 0}
  - Takedowns: {row['winner_name']} {int(row['winner_takedowns']) if pd.notna(row['winner_takedowns']) else 0},  {row['loser_name']} {int(row['loser_takedowns']) if pd.notna(row['loser_takedowns']) else 0}
  """
  if pd.notna(row['bonus']) and row['bonus'] != '-':
    doc += f"Bonus: {row['bonus']}"

  return doc.strip()


print("generating fight documents")

for idx, row in tqdm(fights_df.iterrows(), total=len(fights_df)):
  doc_text = create_fight_document(row)

  safe_name = f"{row['winner_name']}_vs_{row['loser_name']}_{row['fight_id'][:8]}".replace(" ", "_").replace("/", "-")
  file_path = DOCS_DIR / "fights" / f"{safe_name}.txt"

  with open(file_path, "w", encoding='utf-8') as f:
    f.write(doc_text)











generating fight documents


100%|██████████| 8436/8436 [01:26<00:00, 97.10it/s]


In [44]:
print(f"Generated {len(list((DOCS_DIR/'fights').glob('*.txt')))} fight documents")

Generated 8436 fight documents


In [59]:
events_df = conn.execute(
"""
SELECT e.event_id, e.name as event_name,
e.date as event_date,
e.location as event_location,
COUNT(fi.fight_id) as num_fights,
STRING_AGG(DISTINCT fi.weight_class, ', ') as weight_classes
FROM events e
LEFT JOIN fights fi ON e.event_id = fi.event_id
GROUP BY 1,2,3,4
ORDER BY e.date DESC
"""

).fetchdf()



def create_event_document(row, fights_for_events):
  doc  = f"""
    Event : {row['event_name']}
    Date : {row['event_date'].strftime("%B %d, %Y") if pd.notna(row['event_date']) else 'Unknown'}
    Location: {row['event_location']}
    Number of Fights: {row['num_fights']}
    Weight Classes: {row['weight_classes']}

    Fight Results:
  """

  for _, fight in fights_for_events.iterrows():
    doc += f'- {fight['winner_name']} def. {fight['loser_name']} via {fight['method']}'
    if pd.notna(fight['round']):
      doc += f"(R{int(fight['round'])})"
    doc += '\n'
  return doc.strip()


for idx, event in tqdm(events_df.iterrows(), total=len(events_df)):
  event_fights=  fights_df[fights_df['event_name'] == event['event_name']]
  doc_text = create_event_document(event, event_fights)

  safe_name = event["event_name"].replace(" ", "_").replace(":", "").replace("/", "-")[:80]
  file_path = DOCS_DIR / 'events'/ f"{safe_name}.txt"

  with open(file_path, "w", encoding='utf-8') as f:
    f.write(doc_text)

print(f"Done! generated {len(list((DOCS_DIR/'events').glob('*.txt')))} ")










100%|██████████| 768/768 [00:08<00:00, 94.62it/s]

Done! generated 768 


In [71]:
fighters_stats = conn.execute(
    """
SELECT
  f.fighter_id,
  f.name,
  f.dob,
  COUNT(*) as total_fights,
  SUM(CASE WHEN fp.is_winner = TRUE THEN 1 ELSE 0 END) as wins,
  SUM(CASE WHEN fp.is_winner = FALSE THEN 1 ELSE 0 END) as losses,
  SUM(fp.sig_strikes_landed) as total_sig_strikes,
  SUM(fp.takedowns_landed) as total_takedowns,
  SUM(fp.knockdowns) as total_knockdowns,
  SUM(fp.submission_attempts) as total_sub_attempts,
  MIN(e.date) as first_fight,
  MAX(e.date) as last_fight
from fighters f
JOIN fight_performances fp ON f.fighter_id = fp.fighter_id
JOIN fights fi ON fi.fight_id = fp.fight_id
JOIN events e ON fi.event_id = e.event_id
GROUP BY 1,2,3
ORDER BY total_fights DESC


  """
).fetchdf()



def create_fighter_document(row):
  wins = int(row['wins']) if pd.notna(row['wins']) else 0
  losses = int(row['losses']) if pd.notna(row['losses']) else 0

  doc = f"""
  Fighter: {row['name']}
  Date of Birth: {row['dob'].strftime('%Y-%m-%d') if pd.notna(row['dob']) else 'Unknown'}
  Career record: {int(row['total_fights'])}
  First fight: {row['first_fight'].strftime('%Y-%m-%d') if pd.notna(row['first_fight']) else 'Unknown'}
  Last fight: {row['last_fight'].strftime('%Y-%m-%d') if pd.notna(row['last_fight']) else 'Unknown'}

Career Statistics:
- Significant Strikes Landed :{int(row['total_sig_strikes']) if pd.notna(row['total_sig_strikes']) else 0}
- Total takedowns landed :{int(row['total_takedowns']) if pd.notna(row['total_takedowns']) else 0}
- Knockdowns :{int(row['total_knockdowns']) if pd.notna(row['total_knockdowns']) else 0}
- Submission Attempts :{int(row['total_sub_attempts']) if pd.notna(row['total_sub_attempts']) else 0}
  """
  return doc.strip()


for idx, row in tqdm(fighters_stats.iterrows(), total=len(fighters_stats)):
  doc_text = create_fighter_document(row)

  safe_name = row['name'].replace(" ", "_").replace(":", "").replace("/", "-")[:60]
  file_path = DOCS_DIR/"fighters"/f"{safe_name}.txt"

  with open(file_path, 'w', encoding='utf-8') as f:
    f.write(doc_text)


print(f"Generated {len(list((DOCS_DIR/'fighters').glob("*.txt")))}")





100%|██████████| 2685/2685 [00:25<00:00, 105.23it/s]


Generated 2685
